# REST Finalizer and Behavior Statistics

**Two independent analyses:**

- **Behavior:** REST measures from `behavior_review_annotations.csv`, independent of manual REST-segment QC and processed Zarr availability. Missing files remain visible in diagnostics. If the full recording duration is unknown, the REST percentage is `NaN`. Unlabeled time is **not REST** and must not be inferred to be RUN.
- **REST QC / Finalizer:** uses only `rest_segment_rescue_review.csv`, with no legacy QC or fallback. `missing_data` is reported; intervals lacking usable processed Zarr are not exported for Step6b/Step7. Invalid or incomplete manual QC decisions prevent final export.

**Sections 1–11 only inspect and calculate data. Section 12 can write CSV files when `SAVE_FINAL_TABLES` is `True`.** The shipped notebook currently sets this flag to `True`; review the outputs before executing the export cell.

**Scientific exclusions:** `analysis_exclusions` in the shared YAML is applied before behavior statistics and REST finalization. Exclusion provenance remains in diagnostics. Protocol rules follow the project's `variant_keys.protocol` parsing.

**Configuration and paths:** Set `CONFIG_PATH` to the working YAML file. Keeping the YAML with the analysis project (e.g., in `project.analysis_parent_dir`) is recommended for reproducibility, but is not a technical requirement. Source imports assume a notebook launched with the working directory `project/notebook/` and Python modules in `project/src/`.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CONFIG_PATH = Path(r"config.yaml")
# Expected layout:
# project/notebook/REST_Finalizer_and_Behavior_Statistics.ipynb
# project/src/rest_finalizer.py + analysis_exclusions.py + common.py
# The Jupyter working directory must be project/notebook/.
NOTEBOOK_DIR = Path.cwd().resolve()
SRC_DIR = (NOTEBOOK_DIR.parent / "src").resolve()
required_files = ("common.py", "rest_finalizer.py", "analysis_exclusions.py")
missing_files = [name for name in required_files if not (SRC_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Could not find source files: {', '.join(missing_files)}. "
        f"Expected src/ directory: {SRC_DIR}. "
        "Run this notebook from the project's notebook/ directory."
    )
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rest_finalizer import (
    load_inputs, inspect_index, summarize_behavior, behavior_comparisons,
    rest_retention, select_final_cohort, finalize_rest, qc_statistics,
    export_authoritative, STATUS_COLUMNS,
)
from analysis_exclusions import build_exclusion_registry

# The day order and REST thresholds are local notebook settings, not rest_finalizer.py defaults.
DAY_ORDER = ['Before-PT', 'Day-PT', '1-5h-PT', '1d-PT', '3d-PT']
REST_THRESHOLDS_SEC = [30, 60, 120, 180, 300]

# Excluding UNASSIGNED from group statistics is distinct from scientific
# exclusion rules in YAML. The default False retains UNASSIGNED in summaries
# unless a separate scientific exclusion applies.
EXCLUDE_UNASSIGNED_FROM_ANALYSIS = False
USE_ANALYSIS_SELECTION_FOR_QC = False  # Global cohort filters are disabled for REST QC by default.
SAVE_FINAL_TABLES = True  # Writes final CSVs if export checks pass; change to False for preview-only.

print('Config:', CONFIG_PATH)
print('Days:', DAY_ORDER)
print('REST thresholds (sec):', REST_THRESHOLDS_SEC)
print('Exclude UNASSIGNED from group analyses:', EXCLUDE_UNASSIGNED_FROM_ANALYSIS)
print('Save final tables:', SAVE_FINAL_TABLES)


## 1. Input files and experimental group assignment

Read the current Step4 index, `behavior_review_annotations.csv`, and `rest_segment_rescue_review.csv`; the legacy `rest_segment_qc.csv` is not consulted. Experimental group membership comes **only** from `analysis_selection.experimental_groups` in the YAML, not from `analysis_selection.groups`. Unassigned mice remain visible as `UNASSIGNED`.


In [ ]:
src = load_inputs(CONFIG_PATH)
index, config = src['index'], src['config']
print('REST stems in current Step4 index:', len(index))
print('Behavior annotation rows:', len(src['annotations']))
print('REST Review rows:', len(src['review']))
print('Groups:', config.get('analysis_selection',{}).get('experimental_groups',{}))
display(index[['stem','mouse_id','exp_day','state','protocol','experimental_group']].head(20))

## 2. Availability of processed and raw Zarr

This section **does not exclude recordings**. It reports `ok`, `missing`, `signal_missing`, open errors, frame counts, and spatial dimensions. When the processed Zarr cannot provide the recording duration, the raw Zarr is inspected. If neither provides a valid duration, `% REST` cannot be calculated. Missing files do not by themselves terminate diagnostic inspection.


In [ ]:
checked = inspect_index(index, config)
cols = ['stem','exp_day','state','processed_status','raw_status','n_frames','height','width','recording_sec','duration_source','inspection_error']
print('Processed status counts:')
display(checked['processed_status'].value_counts(dropna=False))
display(checked[cols].sort_values(['processed_status','stem']))
print('Missing/unusable processed:', int(checked['processed_status'].ne('ok').sum()))

## 2a. Unified scientific exclusions from YAML

**Apply exclusions after loading metadata and before computing scientific summaries.** Rules are evaluated against the **full Step2 `metadata_registration.csv`** (including stimulation and other protocols), whereas this notebook finalizes only REST-state recordings. `analysis_exclusions.rules` matches exact `stem` values and/or combinations of `mouse_id`, `exp_day`, `state`, and `protocol`. Protocol values use the same `variant_keys.protocol` parsing as Step2; this does not reselect protocol repeats. Fields within a rule are combined with AND; a list of acceptable values for one field uses OR.

`UNASSIGNED` denotes missing experimental group assignment, **not a scientific exclusion**. Scientific exclusions apply regardless of group. Original annotations are not changed. Excluded stems remain in diagnostics and `rest_stem_status.csv`, but contribute neither behavior statistics nor FC-ready segments. By default, enabled rules that match no stems raise an error (`analysis_exclusions.fail_on_unmatched_rules: true` unless configured otherwise).


In [ ]:
# Apply rules to the COMPLETE Step2 registry, including Stim and other protocols.
# Otherwise a valid Stim rule could be reported as unmatched:
# load_inputs() intentionally retains only Rest-state recordings.
metadata_path = Path(src['paths']['metadata_registration_csv'])
step2_metadata = pd.read_csv(metadata_path, dtype=object, keep_default_na=False)
exclusions = build_exclusion_registry(step2_metadata, src['annotations'], config)
exclusion_registry = exclusions['records']
exclusion_rule_report = exclusions['rule_report']
excluded_stems = set(exclusion_registry.loc[exclusion_registry['analysis_excluded'], 'stem'])
checked_analysis = checked.loc[~checked['stem'].astype(str).isin(excluded_stems)].copy()
annotations_analysis = src['annotations'].loc[
    ~src['annotations']['stem'].astype(str).isin(excluded_stems)
].copy() if 'stem' in src['annotations'] else src['annotations'].copy()
excluded_index = checked.loc[checked['stem'].astype(str).isin(excluded_stems)].copy()

print('Active exclusion rules:', len(exclusion_rule_report))
print('Excluded REST stems in index:', len(excluded_index))
print('Excluded stems among ALL Step2 / annotation records:',
      int(exclusion_registry['analysis_excluded'].sum()))
display(exclusion_rule_report)
display(exclusion_registry.loc[exclusion_registry['analysis_excluded'],
    ['stem','mouse_id','exp_day','state','protocol','source','exclusion_rule_ids','exclusion_reason']])
print('Original annotations and manual REST QC remain UNCHANGED.')

## 3. Behavior: two measures per recording

First generate a diagnostic table containing all annotated recordings. Scientific summaries use **only recordings without `analysis_excluded`**; `UNASSIGNED` mice can also be excluded from aggregation using the notebook flag. Neither source annotations nor the original REST interval times are modified.

The measures are REST percentage of the **entire recording** and mean continuous REST period duration (seconds). The recording duration comes from inspected processed/raw Zarr, not from the final annotated boundary.


In [ ]:
# Diagnose ALL original recordings, including scientifically excluded stems.
behavior_all = summarize_behavior(src['annotations'], checked, config, exclude_unassigned=False)
behavior_diag_stem = behavior_all['stem'].merge(
    exclusion_registry[['stem','analysis_excluded','exclusion_rule_ids','exclusion_reason']],
    on='stem', how='left', validate='one_to_one',
)
behavior_diag_stem['analysis_excluded'] = behavior_diag_stem['analysis_excluded'].fillna(False).astype(bool)

# Use only this scientifically eligible subset for means, SD, paired tests, and retention.
behavior = summarize_behavior(
    annotations_analysis, checked_analysis, config,
    exclude_unassigned=EXCLUDE_UNASSIGNED_FROM_ANALYSIS,
)
stem_cols = ['stem','mouse_id','exp_day','state','experimental_group','recording_sec',
             'n_rest_periods','mean_rest_period_sec','rest_percent_recording',
             'behavior_status','behavior_issue','analysis_excluded','exclusion_rule_ids','exclusion_reason']
print('ALL annotated stems (diagnostics):')
display(behavior_diag_stem[stem_cols].sort_values(['exp_day','mouse_id']))
print('Included behavior statuses:', behavior['stem']['behavior_status'].value_counts(dropna=False).to_dict()
      if not behavior['stem'].empty else {})
print('Mouse × day × state for analysis (excluded stems absent):')
if not behavior['mouse_day'].empty:
    display(behavior['mouse_day'].sort_values(['exp_day','mouse_id']))
else:
    print('No eligible mouse-days.')


## 4. Before-PT: descriptive statistics for Awake mice

The statistical unit is the mouse. For each measure, report the valid count, mean, SD, SEM, median, 25th–75th percentiles, minimum, and maximum. A mouse with no REST may contribute a REST percentage of 0% when recording duration is available; its mean REST-period duration is undefined.

`Before-PT` is the actual label used by this notebook, not a universal label required by the pipeline.


In [ ]:
comparisons = behavior_comparisons(behavior['mouse_day'], days=DAY_ORDER)
display(comparisons['before_description'].round(3))
print('Individual Before-PT mouse values:')
display(behavior['mouse_day'].query('exp_day == "Before-PT" and state == "Awake"')
        [['mouse_id','experimental_group','n_rest_periods',
          'mean_rest_period_sec','rest_percent_recording']])

## 5. Recording repeatability: Before-PT versus Day-PT (paired comparison)

Include only mice with **actual `state=Awake` recordings on Day-PT**. A mouse with a Day-PT Iso recording is excluded from **both sides of the pair**, not automatically from unrelated descriptive analyses. This is different from excluding an entire experimental group named `Iso_PT`: the rule uses recorded state. Each mouse contributes one pair; differences are `Day-PT − Before-PT`.

These day labels are specific to this notebook's analysis setup.


In [ ]:
print('Mice excluded from the paired comparison because Day-PT was Iso:')
display(comparisons['excluded_from_pair'])
paired = comparisons['paired']
print('Paired mouse count:', len(paired))
if not paired.empty:
    display(paired[['mouse_id','experimental_group_before',
        'rest_percent_recording_before','rest_percent_recording_daypt','delta_rest_percent_recording',
        'mean_rest_period_sec_before','mean_rest_period_sec_daypt','delta_mean_rest_period_sec']].round(3))
display(comparisons['paired_summary'].round(3))

In [ ]:
if not paired.empty:
    fig, axes = plt.subplots(1,2,figsize=(11,4))
    for ax, metric, label in [(axes[0],'rest_percent_recording','REST (% of recording)'),
                             (axes[1],'mean_rest_period_sec','Mean REST period (s)')]:
        for _,r in paired.iterrows():
            ax.plot([0,1],[r[f'{metric}_before'],r[f'{metric}_daypt']],marker='o',alpha=.65)
        ax.set_xticks([0,1],['Before-PT','Day-PT']);ax.set_ylabel(label);ax.grid(alpha=.2)
    fig.suptitle('Within-mouse repeatability, only Day-PT Awake');plt.tight_layout();plt.show()

## 6. REST dynamics after photothrombosis: descriptive statistics

Plot between-**mouse** mean and SD for the days listed in the notebook's `DAY_ORDER`. The provided order is `Before-PT`, `Day-PT`, `1-5h-PT`, `1d-PT`, and `3d-PT`. Report pooled results and results by the experimental groups configured in YAML (for example, `Control`, `Awake_PT`, `Iso_PT`, if those group names are present). Only records actually acquired in the Awake state enter these summaries. `UNASSIGNED` is excluded only when the corresponding notebook flag is enabled. Each day has its own observed sample size (`n`). This section does **not** calculate each mouse's change from baseline.


In [ ]:
print('Overall by day:')
display(comparisons['day_summary'].round(3))
print('Experimental group × day:')
display(comparisons['group_day_summary'].round(3))

In [ ]:
md = behavior['mouse_day']
awake = md[md['state'].astype(str).str.lower().eq('awake')].copy() if not md.empty else pd.DataFrame()
if not awake.empty:
    x = {day:i for i,day in enumerate(DAY_ORDER)}
    for metric,label in [('rest_percent_recording','REST (% of full recording)'),
                         ('mean_rest_period_sec','Mean continuous REST period (s)')]:
        fig, ax = plt.subplots(figsize=(8.5, 4.5))
        for mouse,sub in awake.groupby('mouse_id'):
            part=sub[sub['exp_day'].isin(x)].copy().sort_values('exp_day',key=lambda s:s.map(x))
            ax.plot(part['exp_day'].map(x), part[metric], alpha=.18, marker='o', lw=1,
                    color='gray', zorder=1)
        summary=awake.groupby('exp_day')[metric].agg(['mean','std','count'])
        days=[d for d in DAY_ORDER if d in summary.index]
        ax.errorbar([x[d] for d in days],[summary.loc[d,'mean'] for d in days],
                    yerr=[summary.loc[d,'std'] if np.isfinite(summary.loc[d,'std']) else 0 for d in days],
                    marker='o',linewidth=2.2,capsize=4,label='All groups: mean ± SD between mice',zorder=2)
        for d in days:
            ax.annotate('n='+str(int(summary.loc[d,'count'])),(x[d],summary.loc[d,'mean']),
                        xytext=(0,8),textcoords='offset points',ha='center',fontsize=8)
        ax.set_xticks(range(len(DAY_ORDER)),DAY_ORDER)
        ax.set_ylabel(label);ax.grid(alpha=.25);ax.legend()
        plt.tight_layout();plt.show()
else:
    print('No eligible Awake behavior rows.')

### Experimental-group trajectories: one mean curve per configured group

For each behavior measure, show an independent mean trajectory (± between-mouse SD) for each group from `analysis_selection.experimental_groups` in YAML. In the provided configuration these may include `Control`, `Awake_PT`, and `Iso_PT`; these are **example group labels**, not fixed requirements of the algorithm. Missing group/day combinations are not interpolated. Each point is annotated with the number of mice (`n`).


In [ ]:
from matplotlib.lines import Line2D

groups_config = config.get('analysis_selection', {}).get('experimental_groups', {})
group_order = list(groups_config)  # Only groups declared in YAML, excluding UNASSIGNED.
md = behavior['mouse_day']
awake = md.loc[md['state'].astype(str).str.lower().eq('awake')].copy() if not md.empty else pd.DataFrame()

if awake.empty:
    print('No eligible Awake behavior rows.')
else:
    positions = {day:i for i,day in enumerate(DAY_ORDER)}
    for metric,label in [('rest_percent_recording','REST (% of full recording)'),
                         ('mean_rest_period_sec','Mean continuous REST period (s)')]:
        fig, ax = plt.subplots(figsize=(9,5))
        for group in group_order:
            part=awake.loc[awake['experimental_group'].eq(group) & awake['exp_day'].isin(DAY_ORDER)]
            if part.empty:
                print(f'{group}: no eligible observations for {metric}')
                continue
            summary=part.groupby('exp_day')[metric].agg(['mean','std','count'])
            days=[d for d in DAY_ORDER if d in summary.index and int(summary.loc[d,'count'])>0]
            if not days: continue
            xs=[positions[d] for d in days]
            ys=[summary.loc[d,'mean'] for d in days]
            err=[summary.loc[d,'std'] if np.isfinite(summary.loc[d,'std']) else 0 for d in days]
            line=ax.errorbar(xs,ys,yerr=err,marker='o',linewidth=2,capsize=4,
                             label=group+' (mean ± SD)')
            color=line[0].get_color()
            for xx, yy, d in zip(xs,ys,days):
                ax.annotate('n='+str(int(summary.loc[d,'count'])),(xx,yy),
                            xytext=(0,8),textcoords='offset points',ha='center',fontsize=8,color=color)
        ax.set_xticks(range(len(DAY_ORDER)),DAY_ORDER)
        ax.set_ylabel(label);ax.grid(alpha=.25);ax.legend()
        plt.tight_layout();plt.show()

## 7. Choosing a minimum continuous REST duration

The threshold list `REST_THRESHOLDS_SEC` (currently `[30, 60, 120, 180, 300]` seconds) is defined **in the first code cell of this notebook**, not in YAML and not in `rest_finalizer.py`. Before manual QC, a mouse/day is retained if at least one continuous annotated REST interval meets the threshold. Report retention across days and groups. Manual 60-second candidate QC is assessed separately below.


In [ ]:
retention = rest_retention(behavior, thresholds=REST_THRESHOLDS_SEC)
print('All mouse-days and by day:')
display(retention[retention['scope'].isin(['overall','day'])].sort_values(['scope','exp_day','min_rest_sec']).round(2))
print('Per group × day:')
display(retention[retention['scope'].eq('group_day')].sort_values(['exp_day','experimental_group','min_rest_sec']).round(2))

In [ ]:
overall=retention[retention['scope'].eq('overall')].sort_values('min_rest_sec')
if not overall.empty:
    fig, ax=plt.subplots(figsize=(7,4));ax.plot(overall['min_rest_sec'],overall['retained_percent'],marker='o')
    for _,r in overall.iterrows():
        ax.annotate(f"{r['n_retained']}/{r['n_mouse_days']}",(r['min_rest_sec'],r['retained_percent']),xytext=(0,8),
                    textcoords='offset points',ha='center')
    ax.set(xlabel='Minimum continuous REST (s)',ylabel='Mouse-days retained (%)',ylim=(0,106))
    ax.grid(alpha=.2);plt.show()

## 8. Diagnostic cohort for REST QC

Scientific `analysis_exclusions` are applied **before** selecting the REST QC cohort. The global `analysis_selection` filters are disabled for QC by default (`USE_ANALYSIS_SELECTION_FOR_QC = False`); applying them could omit Iso or `UNASSIGNED` recordings from diagnostics. Scientifically excluded stems remain in a separate diagnostic table but do not require completed REST QC and do not block final export.


In [ ]:
selection_qc = select_final_cohort(
    checked_analysis, src['annotations'], src['review'], config,
    apply_analysis_selection=USE_ANALYSIS_SELECTION_FOR_QC,
)
qc_cohort = selection_qc['cohort']

# Diagnostic rows of scientific exclusions deliberately outside the QC cohort.
excluded_qc_diag = excluded_index[['stem','mouse_id','exp_day','state','processed_status']].copy()
excluded_qc_diag['in_analysis_selection'] = False
excluded_qc_diag['n_nominal_candidates'] = 0  # Not recomputed for scientifically excluded recordings.
excluded_qc_diag['has_rest_review'] = excluded_qc_diag['stem'].astype(str).isin(
    set(src['review']['stem'].astype(str)) if 'stem' in src['review'] else set())
excluded_qc_diag['in_final_cohort'] = False
excluded_qc_diag = excluded_qc_diag.merge(
    exclusion_registry[['stem','exclusion_rule_ids','exclusion_reason']],
    on='stem', how='left', validate='one_to_one',
)
excluded_qc_diag['reason'] = 'analysis_excluded: ' + excluded_qc_diag['exclusion_rule_ids']
qc_diagnostics = pd.concat([selection_qc['diagnostics'], excluded_qc_diag], ignore_index=True)
print('Full diagnostic REST index:', len(checked))
print('Scientific exclusions:', len(excluded_index))
print('Finalizer review cohort:', len(qc_cohort))
print('Reviewed but outside *eligible* cohort:', len(selection_qc['reviewed_outside_cohort']))
display(qc_diagnostics[['stem','processed_status','in_analysis_selection',
    'n_nominal_candidates','has_rest_review','in_final_cohort','reason']].sort_values('stem'))
if not selection_qc['reviewed_outside_cohort'].empty:
    display(selection_qc['reviewed_outside_cohort'])


## 9. Validating manual REST QC decisions

Only `rest_segment_rescue_review.csv` is used. Scientifically excluded stems are not finalized, but are represented in `rest_stem_status.csv` with `final_state=analysis_excluded`. Existing `USE / REJECT / UNCLEAR` decisions in the manual review CSV remain untouched. Missing-QC export blockers apply to scientifically eligible recordings only.


In [ ]:
final = finalize_rest(qc_cohort, src['annotations'], src['review'], config)

# Preserve the provenance of scientific exclusions in the final status table,
# without inventing REST windows or changing manual annotation files.
review_by_stem = (
    src['review'].drop_duplicates('stem', keep='last').set_index('stem')
    if 'stem' in src['review'] and not src['review'].empty else pd.DataFrame()
)
excluded_status = []
for _, row in excluded_index.iterrows():
    stem = str(row['stem'])
    match = exclusion_registry.loc[exclusion_registry['stem'].eq(stem)].iloc[0]
    st = {col: '' for col in STATUS_COLUMNS}
    for col in ('stem','mouse_id','experimental_group','exp_day','state','protocol',
                'processed_status','processed_path','n_frames'):
        st[col] = row.get(col, '')
    for col in ('n_candidates','n_reviewed','n_accepted','n_rejected','n_unclear','n_overlaps'):
        st[col] = 0
    st['behavior_rest_sec'] = np.nan
    st['review_status'] = review_by_stem.loc[stem, 'review_status'] if stem in review_by_stem.index else ''
    st['final_state'] = 'analysis_excluded'
    st['reason'] = str(match['exclusion_rule_ids']) + ': ' + str(match['exclusion_reason'])
    excluded_status.append(st)
if excluded_status:
    final['rest_stem_status'] = pd.concat([
        final['rest_stem_status'], pd.DataFrame(excluded_status, columns=STATUS_COLUMNS)
    ], ignore_index=True)

# Preserve explicit statuses and provenance for ALL REST stems, not only exclusions.
final['rest_stem_status'] = final['rest_stem_status'].merge(
    exclusion_registry[['stem','analysis_excluded','exclusion_rule_ids','exclusion_reason']],
    on='stem', how='left', validate='one_to_one',
)
final['rest_stem_status']['analysis_excluded'] = (
    final['rest_stem_status']['analysis_excluded'].fillna(False).astype(bool)
)

print('REST finalization states (including excluded):')
display(final['rest_stem_status']['final_state'].value_counts(dropna=False))
print('Records requiring attention, excluding deliberate scientific exclusions:')
attention = final['rest_stem_status'].loc[
    ~final['rest_stem_status']['final_state'].isin({'complete','analysis_excluded','not_applicable_anest'})
]
display(attention[['stem','exp_day','state','processed_status','n_candidates','n_reviewed',
                   'n_accepted','final_state','reason']])
print('Approved 60-s windows with usable processed Zarr:', len(final['rest_segments_final']))
print('Shadow mask status rows (QC cohort only):', len(final['rest_shadow_masks_by_stem']))


## 10. Retention after manual REST QC

QC denominators include **only scientifically eligible** recordings. Excluded stems remain visible in `rest_stem_status.csv`, but are not counted as dropped mice or included in group-level PASS/retention rates. `UNASSIGNED` remains a separate experimental-group status.


In [ ]:
# Keep the analysis_excluded diagnostic state in exported status tables,
# but omit excluded records from QC denominators by using the eligible subset.
final_for_qc = dict(final)
final_for_qc['rest_stem_status'] = final['rest_stem_status'].loc[
    final['rest_stem_status']['final_state'].ne('analysis_excluded')
].copy()
qc = qc_statistics(final_for_qc, exclude_unassigned=EXCLUDE_UNASSIGNED_FROM_ANALYSIS)
if not qc['mouse_day'].empty:
    display(qc['mouse_day'].sort_values(['exp_day','experimental_group','mouse_id']))
if not qc['group_day'].empty:
    display(qc['group_day'].sort_values(['exp_day','experimental_group']))
print('Decisions and usable segments by scientifically eligible stem:')
display(qc['stem'][['stem','experimental_group','exp_day','state','final_state',
    'n_candidates','n_accepted','n_rejected','n_unclear','n_usable_60s','kept_for_fc']])
print('Scientific exclusions omitted from QC denominators:')
display(final['rest_stem_status'].loc[
    final['rest_stem_status']['final_state'].eq('analysis_excluded'),
    ['stem','experimental_group','exp_day','state','final_state','reason']])


## 11. Final tables for Step6b and Step7

- `rest_segments_final.csv`: validated manually accepted `USE` 60-second intervals from scientifically eligible recordings with usable processed data.
- `rest_stem_status.csv`: QC statuses for eligible records **and** scientific exclusions, with `analysis_excluded`, `exclusion_rule_ids`, and `exclusion_reason` provenance.
- `rest_shadow_masks_by_stem.csv`: stem-to-shadow-mask mapping for the scientifically eligible QC cohort. Existing manually saved masks are not deleted.

For later steps, the exclusion rules remain defined by `analysis_exclusions` in YAML and processed by `analysis_exclusions.py`. The frozen REST CSVs alone do not convey exclusions for unrelated recording protocols, such as Stim.


In [ ]:
display(final['rest_segments_final'].head(30))
display(final['rest_shadow_masks_by_stem'].sort_values('stem'))
print('Status counts:')
display(final['rest_stem_status']['final_state'].value_counts(dropna=False))
print('Missing processed (not exported as FC segments):',
      int(final['rest_stem_status']['final_state'].eq('missing_data').sum()))

## 12. Exporting final tables — explicit write operation

**Attention: this notebook currently sets `SAVE_FINAL_TABLES = True` in its first code cell.** After reviewing the exclusions, QC decisions, and status tables, execute the export cell only when you intend to save (or replace) the final CSVs. Setting the flag to `False` gives preview-only behavior; it is **not** the current configured value.

`analysis_excluded` does not block export but remains in the status table and produces no FC-ready segments. Incomplete QC for non-excluded recordings blocks export. Original manual behavior and REST-Review CSVs are not edited. Step7 should use the frozen REST intervals and the same scientific-exclusion rules from YAML.


In [ ]:
blocking_states = {'invalid','missing_review','incomplete_review'}
blockers = final['rest_stem_status'][final['rest_stem_status']['final_state'].isin(blocking_states)]
missing = final['rest_stem_status'][final['rest_stem_status']['final_state'].eq('missing_data')]
shadow_blockers = final['rest_shadow_masks_by_stem'][final['rest_shadow_masks_by_stem']['shadow_status'].isin({'missing','invalid_shape'})]
if not blockers.empty:
    print('NOT SAVED: QC requires attention; inspect rows below:')
    display(blockers[['stem','final_state','reason']])
elif not shadow_blockers.empty:
    print('NOT SAVED: missing/invalid shadow masks; inspect rows below:')
    display(shadow_blockers[['stem','shadow_status','reason']])
elif not SAVE_FINAL_TABLES:
    print('PREVIEW ONLY: authoritative REST CSV NOT modified.')
elif qc_cohort.empty:
    print('NOT SAVED: REST QC cohort is empty.')
elif not selection_qc['reviewed_outside_cohort'].empty:
    print('NOT SAVED: reviewed stems outside current selection; inspect them below.')
    display(selection_qc['reviewed_outside_cohort'])
else:
    if not missing.empty:
        print('ATTENTION: missing processed data; these stems have NO exported FC windows:')
        display(missing[['stem','processed_status','n_accepted','reason']])
    saved = export_authoritative(final,src['root'],save=True)
    print('SAVED:')
    for key,p in saved.items(): print(key,'->',p)

### Important limitations and interpretation

- `% REST` is measured relative to the entire recording duration; unlabeled time must not be interpreted as RUN.
- Before-PT versus Day-PT compares the same mice; Day-PT/Iso is excluded from this pairing only.
- Missing processed Zarr does not prevent diagnostics; a known complete recording duration is required for `% REST`.
- `UNASSIGNED` is lack of experimental group assignment, not the same as scientific exclusion (`analysis_excluded`).
- Cross-analysis exclusion rules are defined by `analysis_exclusions` in YAML. Downstream scripts should use `analysis_exclusions.py`, not copy or reinterpret the matcher.
- Neither the behavior annotation CSV nor the manual REST-Review CSV is modified by these notebook cells.
- The experiment-specific `DAY_ORDER`, `REST_THRESHOLDS_SEC`, and `SAVE_FINAL_TABLES` flags are defined in the first code cell; they are **not** inferred from YAML.
